# LLM

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model

#Groq
groq_llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq"
)
groq_llm.invoke("What is a cat?")

AIMessage(content='A **cat** (scientific name *Felis catus*) is a small, typically furry, carnivorous mammal that is one of the most common domestic pets worldwide. Here are some key points that capture what a cat is:\n\n| Feature | Details |\n|--------|---------|\n| **Taxonomy** | Kingdom: Animalia → Phylum: Chordata → Class: Mammalia → Order: Carnivora → Family: Felidae → Genus: *Felis* → Species: *Felis catus* |\n| **Physical traits** | • Body length 23–48\u202fcm (excluding tail) and a tail about the same length. <br>• Weight ranges from 3–8\u202fkg (average ~4–5\u202fkg). <br>• Sharp retractable claws, highly flexible spine, keen night vision, and excellent hearing. |\n| **Behavior** | • Solitary hunters, though many live in social groups when domesticated. <br>• Known for independent, sometimes aloof behavior but can form strong bonds with humans. <br>• Communicates via purring, hissing, growling, body language, and vocalizations. |\n| **Domestication** | • First domesticated aro

# Setting up the prompt (System Instructions)

In [6]:
prompt = (
        "You are a helpful assistant."
    )

# Setting up the agents

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model=groq_llm,
    tools=[],
    system_prompt=prompt,
    # Remembers the conversation per thread_id (used by telegram_bot.py, one thread per chat)
    # checkpointer=InMemorySaver(),
)

agent.invoke({"messages": [{"role": "user", "content": "What is a cat"}]})


{'messages': [HumanMessage(content='What is a cat', additional_kwargs={}, response_metadata={}, id='d5e7d87f-568b-4d3b-adbc-ceb4bea7b18f'),
  AIMessage(content='A **cat** (scientific name *Felis catus*) is a small, typically furry, carnivorous mammal that is often kept as a pet. Cats are known for:\n\n- **Size**: Usually 2.5–4\u202fkg (5.5–9\u202flb) and about 46–51\u202fcm (18–20\u202fin) long, excluding the tail.\n- **Body**: Slender, flexible, with sharp retractable claws.\n- **Senses**: Excellent night vision, acute hearing, and a highly developed sense of smell.\n- **Behavior**: Independent, curious, and often affectionate. They hunt small prey (e.g., rodents, insects) and are adept climbers.\n- **Reproduction**: Females (queens) can have multiple litters each year, with 3–6 kittens per litter.\n- **Domestication**: Humans have kept cats for companionship and pest control for thousands of years.\n\nCats are beloved worldwide for their playful antics, soothing purring, and the comp

# PREPARING the ML PICKLE FILE

* Getting the DATA
* EDA
* Preprocessing before split
* Split
* Preprocessing after split
* Model

#### First we create our entire pipeline

In [ ]:
import pandas as pd
df = pd.read_csv('kyphosis.csv')

#Preprocess before split
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Kyphosis'] = le.fit_transform(df['Kyphosis'])

#Split
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Kyphosis'])
y = df['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#Preprocess after split
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[['Age']] = scaler.fit_transform(X_train[['Age']])
X_test[['Age']] = scaler.transform(X_test[['Age']])


#------------MODEL-----------------------------

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train,y_train)

preds = dt.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(preds,y_test))



              precision    recall  f1-score   support

           0       0.84      0.80      0.82        20
           1       0.33      0.40      0.36         5

    accuracy                           0.72        25
   macro avg       0.59      0.60      0.59        25
weighted avg       0.74      0.72      0.73        25



### Then we dump our pipeline into pickle files

In [ ]:
dir(scaler)
le.m

array([-0.00515464])

In [ ]:
import pickle

preprocessing = {
    "age_scaler": scaler,
    "label_encoder": le,
}

with open("preprocessing.pkl", "wb") as f:
    pickle.dump(preprocessing, f)

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Testing the pickle files

In [ ]:
model = pickle.load(open("model.pkl", "rb"))
preprocessing = pickle.load(open("preprocessing.pkl", "rb"))

patient = pd.DataFrame({"Age": [15], "Number": [12], "Start": [7]})
patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
prediction = model.predict(patient)
prediction

# Creating a tool for our agent that utilizes the ML pkl file

In [23]:
import pickle
def kyphosis_predict(age:int, number:int,start:int)->int:
    """This tool would be used to predict whether kyphosis is absent or present in a patient.
    It takes Age, Number and start values from user
    age: age of the child in MONTHS.
    number: number of vertebrae involved in the operation.
    start: number of the first vertebra operated on."""
    model = pickle.load(open("model.pkl", "rb"))
    preprocessing = pickle.load(open("preprocessing.pkl", "rb"))

    patient = pd.DataFrame({"Age": [15], "Number": [12], "Start": [7]})
    patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
    prediction = model.predict(patient)
    
    return prediction

kyphosis_predict(12,5,7)
    

array([1])

# Integrating our kyphosis MOdel into our agent

In [20]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model=groq_llm,
    tools=[kyphosis_predict],
    system_prompt="You are a helpful assistant, whenever user gives you info about their childs spinal surgery. Use that to predict whether kyphosis is absent or not, DO NOT do on your own, always use the predict_kyphosis functoin",
    # Remembers the conversation per thread_id (used by telegram_bot.py, one thread per chat)
    # checkpointer=InMemorySaver(),
)

agent.invoke({"messages": [{"role": "user", "content": "A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?"}]})


{'messages': [HumanMessage(content='A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?', additional_kwargs={}, response_metadata={}, id='14e9db32-acc2-45b0-8f97-a904a414db05'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Will they get kyphosis?" We have instructions: "Use that to predict whether kyphosis is absent or not, DO NOT do on your own, always use the predict_kyphosis function." So we must call the function. The function expects age, number, start. The user gave age 71 months, number 3, start 5. So we call the function.', 'tool_calls': [{'id': 'fc_15d28434-7883-4a83-afff-c45bdf491b59', 'function': {'arguments': '{"age":71,"number":3,"start":5}', 'name': 'kyphosis_predict'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 266, 'total_tokens': 386, 'completion_time': 0.133278281, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 